In [8]:
import create_geom_with_pygmsh as cp
import pygmsh

use_pml = True

# Initialize empty geometry using the build in kernel in GMSH
with pygmsh.geo.Geometry() as geom:

    # define global parameters (100km x 20km)
    H = 20 * 1000  # domain height in meter
    L = 100 * 1000  # domain width in meter

    H_w = 4 * 1000  # water depth in meter
    H_g = (H - H_w)  # subsurface depth in meter

    # element size for both water and subsurface domain
    lc_w = 0.15 * 1000.0
    lc_g = 0.25 * 1000.0
    lc_b = min(lc_w, lc_g) # mesh size at the boundary

    H_t = lc_g*2 # thickness of the transition layer

    top_pml = False

    """ node ids without pml layer
    5             6
        layer 3 (water)
    4             3 <--- subsurface topo
        layer 2 (ground)
    4t            3t <--- transition layer
        layer 1 (ground)
    1             2

    # pml layer will be added in create_geom_with_pygmsh.py
    # automatically

    """


    # points
    p1  = (0, 0, 0)
    p2  = (L, 0, 0)
    p3  = (L, H_g, 0)
    p4  = (0, H_g, 0)
    p5  = (0, H, 0)
    p6  = (L, H, 0)
    p3t = (L, H_g - H_t, 0)
    p4t = (0, H_g - H_t, 0)

    # create rectangles for the whole domain
    whole_domain = cp.rectangles(geom)
    # layer 1
    whole_domain.add_one_rect(geom, p1, p2, p3t, p4t, lc_g, transfinite=True, mat_tag="M1")
    # layer 2
    whole_domain.add_one_rect(geom, p4t, p3t, p3, p4, [lc_g, lc_g, lc_w, lc_w], transfinite=False, mat_tag="M1")
    # layer 3
    whole_domain.add_one_rect(geom, p4, p3, p6, p5, lc_w, transfinite=True, mat_tag="M2")

    # create pml layer
    if use_pml:
        lc_pml = min(lc_g, lc_w)
        w_pml = lc_pml * 6
        whole_domain.add_pml_layers(geom, w_pml, lc_pml, top_pml=False)

    # build up edges and lines
    whole_domain.build_points_edges(geom)

    # force recombine all the surfaces
    import gmsh
    gmsh.option.setNumber("Mesh.RecombineAll", 1)
    gmsh.option.setNumber("Mesh.RecombinationAlgorithm", 2)

    # set write
    mesh = geom.generate_mesh(dim=2, order=2, verbose=True, algorithm=8)
    mesh.write("mesh.xdmf")



rectangle id 0 is meshed with nelm_h (bottom,top) = None                 and nelm_v (left, right) = None
rectangle id 1 is meshed with nelm_h (bottom,top) = None                 and nelm_v (left, right) = None
rectangle id 2 is meshed with nelm_h (bottom,top) = None                 and nelm_v (left, right) = None
bottom rectangle id 0
top rectangle id 2
rectangle id 3 is meshed with nelm_h (bottom,top) = [7, 7]                 and nelm_v (left, right) = None
rectangle id 4 is meshed with nelm_h (bottom,top) = [7, 7]                 and nelm_v (left, right) = None
rectangle id 5 is meshed with nelm_h (bottom,top) = [7, 7]                 and nelm_v (left, right) = None
rectangle id 6 is meshed with nelm_h (bottom,top) = [7, 7]                 and nelm_v (left, right) = None
rectangle id 7 is meshed with nelm_h (bottom,top) = [7, 7]                 and nelm_v (left, right) = None
rectangle id 8 is meshed with nelm_h (bottom,top) = [7, 7]                 and nelm_v (left, right) = None
re

In [9]:
from meshio2spec2d import *

mio2spec = Meshio2Specfem2D(mesh)
mio2spec.write("test")

Time elapsed for nodes:  0.21415019035339355  seconds
Time elapsed for mesh:  0.08447575569152832  seconds
material keys:  ['M1', 'M2']
cell_id_offset:  2532
Time elapsed for material:  0.04577445983886719  seconds
Time elapsed for surf:  0.2689845561981201  seconds
Number of PML_X elements:  1128  ->  940  ( 188  elements removed)
Number of PML_Y elements:  1600  ->  1200  ( 400  elements removed)
Number of PML_XY elements:  48  ->  46  ( 2  elements removed)
Time elapsed for cpml:  0.026102781295776367  seconds
Time elapsed:  0.6399350166320801  seconds


In [10]:
# write MESH/nummaterial_velocity_file separately

"""
# format:
#(1)domain_id #(2)material_id #(3)rho #(4)vp #(5)vs #(6)Q_k #(7)Q_mu #(8)ani
#
#  where
#     domain_id          : 1=acoustic / 2=elastic / 3=poroelastic
#     material_id        : POSITIVE integer identifier of material block
#     rho                : density
#     vp                 : P-velocity
#     vs                 : S-velocity
#     Q_k                : 9999 = no Q_kappa attenuation
#     Q_mu               : 9999 = no Q_mu attenuation
#     ani                : 0=no anisotropy/ 1,2,.. check with aniso_model.f90
#
# example:
# 2   1 2300 2800 1500 9999.0 9999.0 0
#
# or
#
#(1)domain_id #(2)material_id  tomography elastic  #(3)filename #(4)positive
#
#  where
#     domain_id : 1=acoustic / 2=elastic / 3=poroelastic
#     material_id        : NEGATIVE integer identifier of material block
#     filename           : filename of the tomography file
#     positive           : a positive unique identifier
#
# example:
# 2  -1 tomography elastic tomo.xyz 1
"""

fname = "MESH/nummaterial_velocity_file"

# test
#sim_water = Specfem2dConfig(100, 4, 5.0, 1.5, 1.5) # lx, lz, freq, v_min, v_max
#sim_earth = Specfem2dConfig(100, 16, 5.0, 1.2, 2.4) # lx, lz, freq, v_min, v_max

# M1: subsurface
M1_params = "2 1 1850.d0 2400.d0 1200.d0 9999 9999 0"

# M2: water
M2_params = "1 2 1020.d0 1500.d0 0.d0 9999 9999 0"

with open(fname, "w") as f:
    f.write(M1_params + "\n")
    f.write(M2_params + "\n")

In [11]:
# prepare MESH_flat/STATIONS

# prepare STATIONS
#n_rec_x = 49
#n_rec_z = 5
l_rec_dx = 2000.0 # in meters
l_rec_dz = 1000.0 # in meters
n_rec_x = int(L / l_rec_dx)
n_rec_z = int(H_w / l_rec_dz) + 1
x_orig = 2000.0 # in meters from the left bound
z_orig = 16000.0 # in meters from the bottom bound

name_station = "DATA/STATIONS"

with open(name_station, "w") as f:

    for i in range(n_rec_x):

        x = x_orig + i * l_rec_dx
        # at j=0, z is always on the ocean bottom
        #z_bottom = my_sinusoidal(x)
        #l_rec_dz = (20000.0 - z_bottom) / (n_rec_z - 1)# place "n_rec_z" receivers equally between the ocean bottom and the top

        z_bottom = z_orig

        for j in range(n_rec_z):
            name_sta = "x_" + str(i)
            name_net = "z_" + str(j)

            z = z_bottom + j * l_rec_dz



            # STATIONS
            # format:
            #   #name #network #x #z #elevation #burial_depth
            f.write("{}\t{}\t{}\t{}\t{}\t{}\n".format(name_sta, name_net, x, z, 0.0, 0.0))

FileNotFoundError: [Errno 2] No such file or directory: 'DATA/STATIONS'